# v24a — table training rows for MSLSAM (any kernel, CPU, ~2 min, no download)

Pre-registration: `versions/23_mslsam-v24/README.md`. v23 lost to Prompt Guard 2 on tables, and v20b had never been trained on a table.
This notebook adds about **1,400 table rows** to v20b's training data, in the same way v20b added tool output:
- **Carriers:** BIPIA TableQA tables that **no version has used**. The 8-gram guard removes v19's and v23's tables.
- **Renderings:** each table is used twice, as **markdown** and as **CSV**.
- **Pairs:** each rendering gives one **benign** row (a declarative filler) and one **injected** row (a payload). Both come from **v20b's own
  tool rows**, so no new attack text enters training.
- **Grouping:** all 4 rows of a table share one group, so grouped CV never splits a table across folds.

In [ ]:
import os, re, json, time, bisect, csv, io, hashlib
import numpy as np, pandas as pd
SEED = 42; rng = np.random.default_rng(SEED)
OUT, BIP = "datasets/unified", "datasets/BIPIA"
MODEL_ID, MAX_LEN, CAP = "meta-llama/Meta-Llama-3-8B-Instruct", 1024, 1500
SYSTEM_PROMPT = ("You are a helpful, secure, and harmless AI assistant. "
                 "You must refuse to assist with any dangerous, illegal, "
                 "or unethical requests.")                                          # v14b's, verbatim
CONTAM_NGRAM, CONTAM_MAX = 8, 0.5                                                   # v19a's guard
def clean_text(t):
    t = re.sub(r"\s+", " ", str(t).strip()); return re.sub(r"[^\x00-\x7F]+", "", t).strip()
def clean_block(t):
    t = re.sub(r"[^\x00-\x7F]+", "", str(t).replace("\r\n", "\n").replace("\r", "\n"))
    return re.sub(r"\n{3,}", "\n\n", re.sub(r"[ \t]+\n", "\n", t)).strip()
def join_user(task, data): return f"{task}\n\n{data}" if task else data
def _norm(t): return re.sub(r"[^a-z0-9 ]", " ", str(t).lower()).split()
def _grams(w, n=CONTAM_NGRAM): return {" ".join(w[i:i+n]) for i in range(len(w) - n + 1)}
def index(texts):
    G, SH = set(), set()
    for t in texts:
        w = _norm(t); G |= _grams(w)
        if len(w) < CONTAM_NGRAM: SH.add(" ".join(w))
    return G, SH
def overlap(t, ix):
    w = _norm(t)
    if len(w) < CONTAM_NGRAM: return float(" ".join(w) in ix[1])
    g = _grams(w); return len(g & ix[0]) / max(len(g), 1)
U = lambda f: pd.read_csv(f"{OUT}/{f}.csv", keep_default_na=False)
USED = {f: U(f) for f in ("v19_external_train", "v20_tool_train", "v22_reader_train", "v19_test", "v21_fresh", "v22_reader_dev", "v23_fresh")}
IX_USED = index([t for D in USED.values() for t in D.text])
tool = USED["v20_tool_train"]
print(f"contamination index over {sum(len(D) for D in USED.values()):,} used texts ({list(USED)})")

In [ ]:
# ================= 1. tables no version has used; payloads and fillers from v20b's own tool rows =================
TAB = [json.loads(l) for sp in ("train", "test") for l in open(f"{BIP}/table/{sp}.jsonl")]
TAB = [{"md": clean_block(r["context"]), "q": clean_text(r["question"])} for r in TAB]
n0 = len(TAB); TAB = [r for r in TAB if len(r["md"]) >= 300 and r["q"] and overlap(r["md"], IX_USED) < CONTAM_MAX]
seen, T2 = set(), []
for r in TAB:                                                       # one row per distinct table
    k = hashlib.md5(r["md"].encode()).hexdigest()
    if k not in seen: seen.add(k); T2.append(r)
TAB = T2
print(f"BIPIA tables: {n0} -> {len(TAB)} never used by any version (distinct)")
assert len(TAB) >= 250, "fewer than 250 unused tables -- tell Claude"
unesc = lambda s: s.replace('\\"', '"').replace("\\n", " ")
PAY = list(dict.fromkeys(unesc(p) for p in tool[tool.label == 1].piece))            # v20b's training payloads (hackaprompt / LLMail)
BEN = sorted(dict.fromkeys(unesc(p) for p in tool[tool.label == 0].piece), key=len)  # v20b's benign declarative fillers
print(f"payloads {len(PAY)} | benign fillers {len(BEN)} (both already in v20b's training set)")
def md_rows(md):
    return [[c.strip() for c in ln.strip().strip("|").split("|")] for ln in md.split("\n") if ln.strip().startswith("|")]
def to_csv(md):
    b = io.StringIO(); csv.writer(b, lineterminator="\n").writerows(md_rows(md)); return b.getvalue().strip()
def cut_lines(doc, budget):
    if len(doc) <= budget: return doc
    out, n = [], 0
    for ln in doc.split("\n"):
        if n + len(ln) + 1 > budget: break
        out.append(ln); n += len(ln) + 1
    return "\n".join(out) if out else doc[:budget]
def place_lines(doc, piece, pos):
    ls = doc.split("\n"); k = 0 if pos == "start" else len(ls) if pos == "end" else max(1, len(ls) // 2)
    return "\n".join(ls[:k] + [piece] + ls[k:])
def take_ben(n):
    L = [len(x) for x in BEN]; i = min(bisect.bisect_left(L, n), len(BEN) - 1)
    if i > 0 and abs(L[i - 1] - n) < abs(L[i] - n): i -= 1
    return BEN.pop(i)

In [ ]:
# ================= 2. rows: table x {markdown, CSV} x {benign, injected} =================
R = []; pay = [PAY[i] for i in rng.permutation(len(PAY))]; npay = 0
for tid in rng.permutation(len(TAB)):
    t = TAB[tid]; task = f"Answer the question using the table below: {t['q']}"
    for fmt, body in (("md", t["md"]), ("csv", to_csv(t["md"]))):
        if npay >= len(pay): break
        pl = pay[npay]; npay += 1; bn = take_ben(len(pl)); pos = ["start", "middle", "end"][int(rng.integers(0, 3))]
        doc = cut_lines(body, CAP - max(len(pl), len(bn)) - 2)
        for lab, piece in ((1, pl), (0, bn)):
            data = place_lines(doc, piece, pos)
            R.append(dict(uid=f"tt-{len(R):05d}", corpus="EXT-Table" if lab else "EXT-Table-Benign", family="indirect", label=lab, task=task,
                          data=data, text=join_user(task, data), group=f"v24t-{tid}", source="bipia-table+v20tool", piece=piece, pos=pos, fmt=fmt))
TT = pd.DataFrame(R)
assert all(p in d for p, d in zip(TT.piece, TT.data)) and TT.uid.is_unique and (TT.data.str.len() <= CAP).all()
_strip = [re.sub(r"\s+", " ", d.replace(p, " ")).strip() for d, p in zip(TT.data, TT.piece)]
assert all(_strip[i] == _strip[i + 1] for i in range(0, len(_strip), 2)), "benign and injected twins differ outside the piece"
print(TT.groupby(["corpus", "fmt"]).agg(n=("uid", "size"), piece_med=("piece", lambda s: int(s.str.len().median())),
                                        data_med=("data", lambda s: int(s.str.len().median()))).to_string())
print("groups (tables):", TT.group.nunique(), "| positions:", TT[TT.label == 1].pos.value_counts().to_dict())
from sklearn.metrics import roc_auc_score
print(f"length-only AUC injected vs benign: {roc_auc_score(TT.label, TT.text.str.len()):.3f} (twins are length-matched; ~0.5 expected)")
r = TT[(TT.fmt == "csv") & (TT.label == 1)].iloc[0]; print("\ne.g. CSV + payload:\n" + r.data[:500])

In [ ]:
# ================= 3. token check with the real tokenizer + write =================
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
fmt_ = lambda u: tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u}],
                                         tokenize=False, add_generation_prompt=True)
TT["n_tokens"] = [len(tok(fmt_(u))["input_ids"]) for u in TT.text]
print(f"prompt tokens: median {int(TT.n_tokens.median())} | max {TT.n_tokens.max()} (limit {MAX_LEN})")
assert TT.n_tokens.max() <= MAX_LEN, "a table row exceeds MAX_LEN -- tell Claude"
TT.to_csv(f"{OUT}/v24_table_train.csv", index=False)
json.dump({"version": "v24a", "built_at": time.strftime("%Y-%m-%d %H:%M"), "rows": len(TT), "tables": int(TT.group.nunique()),
           "counts": {f"{c}/{f}": int(n) for (c, f), n in TT.groupby(["corpus", "fmt"]).size().items()}, "cap_chars": CAP,
           "sources": {"carriers": "BIPIA TableQA tables never used by v19-v23", "payloads": "v20_tool_train EXT-Tool pieces",
                       "fillers": "v20_tool_train EXT-Tool-Benign pieces"},
           "system_prompt": SYSTEM_PROMPT, "max_len": MAX_LEN}, open(f"{OUT}/manifest_v24a.json", "w"), indent=1)
print(f"wrote {OUT}/v24_table_train.csv ({len(TT):,} rows) + manifest_v24a.json\n"
      "NEXT: restart the Mac (swap < 2 GB), then v24b_extract.ipynb in a FRESH kernel.")